<a href="https://colab.research.google.com/github/Vronska-Anhelina/University-app-/blob/main/A_B_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scipy statsmodels openpyxl --quiet
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from google.colab import drive, files
import warnings
warnings.filterwarnings("ignore")


In [2]:
drive.mount("/content/drive")
import zipfile
zip_path = "/content/drive/MyDrive/university_schema.zip"
extract_path = "/content/university_schema"
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)
path = extract_path
users = pd.read_csv(f"{path}/users.csv")
groups = pd.read_csv(f"{path}/groups.csv")
courses = pd.read_csv(f"{path}/courses.csv")
semesters = pd.read_csv(f"{path}/semesters.csv")
teachers = pd.read_csv(f"{path}/teachers.csv")
students = pd.read_csv(f"{path}/students.csv")
rooms = pd.read_csv(f"{path}/rooms.csv")
time_slot = pd.read_csv(f"{path}/time_slot.csv")
course_offerings = pd.read_csv(f"{path}/course_offerings.csv")
lessons = pd.read_csv(f"{path}/lessons.csv")
grades = pd.read_csv(f"{path}/grades.csv")
schedule_entries = pd.read_csv(f"{path}/schedule_entries.csv")
ab_tests = pd.read_csv(f"{path}/ab_tests.csv")

Mounted at /content/drive


In [3]:
data = (
    course_offerings
    .merge(groups[["group_id", "name", "specialty", "year_of_study"]],
           on="group_id", how="left")
    .rename(columns={"name": "group_name"})
    .merge(courses[["course_id", "name"]],
           on="course_id", how="left")
    .rename(columns={"name": "course_name"})
    .merge(teachers[["teacher_id", "name", "cathedra"]],
           on="teacher_id", how="left")
    .rename(columns={"name": "teacher_name"})
    .merge(semesters[["semester_id", "number", "year", "date_start", "date_end"]],
           on="semester_id", how="left")
    .rename(columns={"number": "semester_number", "year": "semester_year"})
)

print(f"Dataset: {len(data)} рядків i колонки: {list(data.columns)}")


Dataset: 1964 рядків i колонки: ['course_offering_id', 'course_id', 'group_id', 'teacher_id', 'semester_id', 'c', 'group_name', 'specialty', 'year_of_study', 'course_name', 'teacher_name', 'cathedra', 'semester_number', 'semester_year', 'date_start', 'date_end']


In [4]:
grades_full = (
    grades
    .merge(students[["student_id", "full_name", "entry_year"]],
           on="student_id", how="left")
    .merge(groups[["group_id", "name", "specialty", "year_of_study"]],
           on="group_id", how="left")
    .rename(columns={"name": "group_name"})
    .merge(course_offerings[["course_offering_id", "course_id", "teacher_id", "semester_id"]],
           on="course_offering_id", how="left")
    .merge(courses[["course_id", "name"]],
           on="course_id", how="left")
    .rename(columns={"name": "course_name"})
    .merge(teachers[["teacher_id", "name", "cathedra"]],
           on="teacher_id", how="left")
    .rename(columns={"name": "teacher_name"})
    .merge(semesters[["semester_id", "number", "year"]],
           on="semester_id", how="left")
    .rename(columns={"number": "semester_number", "year": "semester_year"})
)
print(f"grades_full: {len(grades_full)} рядків i колонки: {list(grades_full.columns)}")

grades_full: 50523 рядків i колонки: ['student_id', 'course_offering_id', 'value', 'date', 'group_id', 'full_name', 'entry_year', 'group_name', 'specialty', 'year_of_study', 'course_id', 'teacher_id', 'semester_id', 'course_name', 'teacher_name', 'cathedra', 'semester_number', 'semester_year']


In [5]:
lessons_full = (
    lessons
    .merge(course_offerings[["course_offering_id", "course_id", "group_id",
                        "teacher_id", "semester_id"]],
           on="course_offering_id", how="left")
    .merge(courses[["course_id", "name"]],
           on="course_id", how="left")
    .rename(columns={"name": "course_name"})
    .merge(groups[["group_id", "name", "specialty"]],
           on="group_id", how="left")
    .rename(columns={"name": "group_name"})
    .merge(teachers[["teacher_id", "name", "cathedra"]],
           on="teacher_id", how="left")
    .rename(columns={"name": "teacher_name"})
    .merge(semesters[["semester_id", "number", "year"]],
           on="semester_id", how="left")
    .rename(columns={"number": "semester_number", "year": "semester_year"})
)
print(f" lessons_full: {len(lessons_full)} рядків i колонки: {list(grades_full.columns)}")


 lessons_full: 33482 рядків i колонки: ['student_id', 'course_offering_id', 'value', 'date', 'group_id', 'full_name', 'entry_year', 'group_name', 'specialty', 'year_of_study', 'course_id', 'teacher_id', 'semester_id', 'course_name', 'teacher_name', 'cathedra', 'semester_number', 'semester_year']


In [6]:
ab_full = (
    ab_tests
    .merge(users[["id", "full_name", "role", "email"]],
           left_on="id_user", right_on="id", how="left")
)
print(f"ab_full: {len(ab_full)} рядків  i колонки: {list(ab_full.columns)}")

ab_full: 7715 рядків  i колонки: ['id_user', 'test_id', 'ab_group', 'id', 'full_name', 'role', 'email']


# **АГРЕГАЦІЯ ДЛЯ TABLEAU**

Середній бал по групах і семестрах

In [7]:
avg_grade_group = (
    grades_full
    .groupby(["group_id", "group_name", "specialty",
              "semester_id", "semester_number", "semester_year"])
    .agg(
        avg_grade     = ("value", "mean"),
        total_grades  = ("value", "count"),
        students_cnt  = ("student_id", "nunique")
    )
    .reset_index())
avg_grade_group.head()

,group_id,group_name,specialty,semester_id,semester_number,semester_year,avg_grade,total_grades,students_cnt
0,1,IPZ-1A,IPZ,1,1,2022,74.6680,100,25
1,1,IPZ-1A,IPZ,2,2,2022,77.3936,125,25
2,1,IPZ-1A,IPZ,3,1,2023,75.7288,125,25
3,1,IPZ-1A,IPZ,4,2,2023,76.8090,100,25
4,1,IPZ-1A,IPZ,5,1,2024,75.1800,100,25


 Навантаження викладачів

In [8]:
teacher_load = (
    data
    .groupby(["teacher_id", "teacher_name", "cathedra",
              "semester_id", "semester_number", "semester_year"])
    .agg(
        courses_count = ("course_offering_id", "count"),
        groups_count  = ("group_id", "nunique")
    )
    .reset_index()
)
teacher_load.head()

,teacher_id,teacher_name,cathedra,semester_id,semester_number,semester_year,courses_count,groups_count
0,1,Василина Таран,Mathematics,1,1,2022,4,4
1,1,Василина Таран,Mathematics,2,2,2022,3,3
2,1,Василина Таран,Mathematics,3,1,2023,6,6
3,1,Василина Таран,Mathematics,4,2,2023,10,10
4,1,Василина Таран,Mathematics,5,1,2024,6,6


Кількість уроків по курсах

In [9]:
lessons_per_course = (
    lessons_full
    .groupby(["course_offering_id", "course_name", "group_name",
              "teacher_name", "semester_number", "semester_year"])
    .agg(
        lessons_count = ("lesson_id", "count"),
        lesson_types  = ("type", lambda x: x.value_counts().to_dict())
    )
    .reset_index()
    .drop(columns=["lesson_types"])
)
lessons_per_course.head()


,course_offering_id,course_name,group_name,teacher_name,semester_number,semester_year,lessons_count
0,1,Computer Graphics,IPZ-1A,пан Андрій Гайворонський,1,2022,14
1,2,Operating Systems,IPZ-1A,Орися Піддубна,1,2022,15
2,3,Information Security,IPZ-1A,пан Андрій Гайворонський,1,2022,14
3,4,Web Development,IPZ-1A,Віолетта Швачка,1,2022,16
4,5,Machine Learning,IPZ-1B,Феофан Данилюк,1,2022,14


Кількість course_offerings по групах

In [10]:
offerings_per_group = (
    data
    .groupby(["group_id", "group_name", "specialty", "year_of_study",
              "semester_id", "semester_number", "semester_year"])
    .agg(
        courses_count   = ("course_offering_id", "count"),
        teachers_count  = ("teacher_id", "nunique"),
        courses_list    = ("course_name", lambda x: ", ".join(x.dropna().unique()))
    )
    .reset_index()
)
offerings_per_group.head()

,group_id,group_name,specialty,year_of_study,semester_id,semester_number,semester_year,courses_count,teachers_count,courses_list
0,1,IPZ-1A,IPZ,1,1,1,2022,4,3,"Computer Graphics, Operating Systems, Informat..."
1,1,IPZ-1A,IPZ,1,2,2,2022,5,5,"Web Development, Information Security, OOP, Al..."
2,1,IPZ-1A,IPZ,1,3,1,2023,5,5,"Computer Graphics, Computer Architecture, Func..."
3,1,IPZ-1A,IPZ,1,4,2,2023,4,4,"Numerical Methods, Computer Graphics, Computer..."
4,1,IPZ-1A,IPZ,1,5,1,2024,4,4,"Computer Graphics, Artificial Intelligence, Co..."


# СТАТИСТИКА

In [11]:
def calc_ab_stats(successes_a, trials_a, successes_b, trials_b):
    rate_a = successes_a / trials_a if trials_a > 0 else 0
    rate_b = successes_b / trials_b if trials_b > 0 else 0

    count  = np.array([successes_a, successes_b])
    nobs   = np.array([trials_a, trials_b])
    _, p   = proportions_ztest(count, nobs)

    ci_a   = proportion_confint(successes_a, trials_a, alpha=0.05)
    ci_b   = proportion_confint(successes_b, trials_b, alpha=0.05)
    uplift = (rate_b - rate_a) / rate_a * 100 if rate_a > 0 else 0

    if p < 0.05:
        verdict = "Sample 2 is more successful"  if rate_b > rate_a else "Sample 1 is more successful"
    else:
        verdict = "No significant difference"

    return {
        "group1_successes": successes_a, "group1_trials": trials_a,
        "group1_rate":      round(rate_a * 100, 3),
        "group2_successes": successes_b, "group2_trials": trials_b,
        "group2_rate":      round(rate_b * 100, 3),
        "relative_uplift":  round(uplift, 2),
        "p_value":          round(p, 5),
        "ci1_low":          round(ci_a[0] * 100, 1),
        "ci1_high":         round(ci_a[1] * 100, 1),
        "ci2_low":          round(ci_b[0] * 100, 1),
        "ci2_high":         round(ci_b[1] * 100, 1),
        "verdict":          verdict,
        "significant":      p < 0.05,
        "confidence_level": "95%"
    }

# ТЕСТ 1.  Оптимізація кількості курсів на групу

In [25]:
per_student = (
    course_offerings.merge(students[["student_id", "group_id"]], on="group_id")
    .groupby("student_id")["course_offering_id"].count().reset_index()
    .rename(columns={"course_offering_id": "count"})
)
dataframe1 = (
    ab_full[ab_full["test_id"] == 1]
    .merge(students[["student_id"]], left_on="id_user", right_on="student_id", how="left")
    .merge(per_student, on="student_id", how="left")
)
dataframe1["count"]   = dataframe1["count"].fillna(0)
dataframe1["is_success"] = (dataframe1["count"] > 0).astype(int)
dataframe1["test_name"]  = "Навігація по курсах(Тест 1)"
dataframe1["metric"]     = "course_offerings > 0"

a_group = dataframe1[dataframe1["ab_group"] == "A"]
b_group = dataframe1[dataframe1["ab_group"] == "B"]
success = calc_ab_stats(a_group["is_success"].sum(), len(a_group), b_group["is_success"].sum(), len(b_group))

# ТЕСТ 2.  Додавання блоку прогресу студента на головній сторінці


In [26]:
avg_grade_student = (
    grades[["student_id", "value"]]
    .groupby("student_id")["value"]
    .mean()
    .reset_index()
    .rename(columns={"value": "avg_grade"})
)
dataframe2 = (
    ab_full[ab_full["test_id"] == 2]
    .merge(students[["student_id"]], left_on="id_user", right_on="student_id", how="left")
    .merge(avg_grade_student, on="student_id", how="left")
)
dataframe2["avg_grade"]  = dataframe2["avg_grade"].fillna(0)
dataframe2["is_success"] = (dataframe2["avg_grade"] >= 75).astype(int)
dataframe2["test_name"]  = "Блок прогресу студента(Тест 2)"
dataframe2["metric"]  = "avg_grade >= 75"

a_group2 = dataframe2[dataframe2["ab_group"] == "A"]
b_group2 = dataframe2[dataframe2["ab_group"] == "B"]
success2 = calc_ab_stats(a_group2["is_success"].sum(), len(a_group2), b_group2["is_success"].sum(), len(b_group2))

# ТЕСТ 3. Оновлення сторінки профілю викладача.


In [28]:
per_teacher = (
    course_offerings.groupby("teacher_id")["course_offering_id"]
    .count().reset_index()
    .rename(columns={"course_offering_id": "count"})
)
dataframe3 = (
    ab_full[ab_full["test_id"] == 3]
    .merge(teachers[["teacher_id"]], left_on="id_user", right_on="teacher_id", how="left")
    .merge(per_teacher, on="teacher_id", how="left")
)
dataframe3["count"]   = dataframe3["count"].fillna(0)
dataframe3["is_success"] = (dataframe3["count"] > 2).astype(int)
dataframe3["test_name"]="Профіль викладача(Тест 3)"
dataframe3["metric"]  = "course_offerings > 2"

a_group3 = dataframe3[dataframe3["ab_group"] == "A"]
b_group3 = dataframe3[dataframe3["ab_group"] == "B"]
success3 = calc_ab_stats(a_group3["is_success"].sum(), len(a_group3), b_group3["is_success"].sum(), len(b_group3))


# Таблиця результатів

In [30]:
ab_summary = pd.DataFrame([
    {"test_id": 1, "test_name": "Навігація по курсах(Тест 1)",   "metric": "course_offerings > 0", **success},
    {"test_id": 2, "test_name": "Блок прогресу студента(Тест 2)","metric": "avg_grade >= 75", **success2},
    {"test_id": 3, "test_name": "Профіль викладача(Тест 3)", "metric": "course_offerings > 2",  **success3},
])
ab_summary.head(3)

,test_id,test_name,metric,group1_successes,group1_trials,group1_rate,group2_successes,group2_trials,group2_rate,relative_uplift,p_value,ci1_low,ci1_high,ci2_low,ci2_high,verdict,significant,confidence_level
0,1,Навігація по курсах(Тест 1),course_offerings > 0,771,771,100.000,772,772,100.000,0.00,NaN,100.0,100.0,100.0,100.0,No significant difference,False,95%
1,2,Блок прогресу студента(Тест 2),avg_grade >= 75,372,771,48.249,368,772,47.668,-1.20,0.81944,44.7,51.8,44.1,51.2,No significant difference,False,95%
2,3,Профіль викладача(Тест 3),course_offerings > 2,37,771,4.799,23,772,2.979,-37.92,0.06450,3.3,6.3,1.8,4.2,No significant difference,False,95%


# РЕЗУЛЬТАТИ

In [31]:
for _, row in ab_summary.iterrows():
    print(f"\nТЕСТ {int(row.test_id)}: {row.test_name}")
    print(f" CR: {row.group1_rate}% → {row.group2_rate}% | uplift: {row.relative_uplift}% | p={row.p_value}")
    print(f"{row.verdict}")



ТЕСТ 1: Навігація по курсах(Тест 1)
 CR: 100.0% → 100.0% | uplift: 0.0% | p=nan
No significant difference

ТЕСТ 2: Блок прогресу студента(Тест 2)
 CR: 48.249% → 47.668% | uplift: -1.2% | p=0.81944
No significant difference

ТЕСТ 3: Профіль викладача(Тест 3)
 CR: 4.799% → 2.979% | uplift: -37.92% | p=0.0645
No significant difference


# Експорт в  EXCEL документ. Далі для роботи в  TABLEAU , створення дешбордів по результатах A\B тестувань (№ 1 - 3)

In [34]:
with pd.ExcelWriter("analytics_testing.xlsx", engine="openpyxl") as writer:
    ab_summary.to_excel(writer, sheet_name="AB_summary",index=False)
    grades_full.to_excel(writer, sheet_name="grades_full",index=False)
    teacher_load.to_excel(writer, sheet_name="teacher_test",index=False)
    data.to_excel(writer, sheet_name="dataset",  index=False)
files.download("analytics_testing.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>